# Wine Recognition - Modelo de clasificación

**Cloud Computing | Kubernetes | Entrega 1**

Entrena un clasificador sobre el dataset katossky/wine-recognition (178 vinos, 13 variables químicas -> `label` 1/2/3)

In [25]:
%pip install -q pandas==3.1.0 scikit-learn==1.9.1 joblib==1.6.0

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
from pathlib import Path
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [27]:
DATA_URL = (
    "https://huggingface.co/datasets/katossky/wine-recognition/resolve/main/wine.csv"
)
df = pd.read_csv(DATA_URL)

## Estudio del conjunto de datos

In [28]:
print("Filas y columnas:", df.shape)
print("Valores nulos:", df.isna().sum().sum())
print("Vinos por tipo:", df["label"].value_counts().sort_index().to_dict())
display(df.describe().T)
display(df.groupby("label").mean().T)

Filas y columnas: (178, 14)
Valores nulos: 0
Vinos por tipo: {1: 59, 2: 71, 3: 48}


,count,mean,std,min,25%,50%,75%,max
label,178.0,1.938202,0.775035,1.00,1.0000,2.000,3.0000,3.00
alcohol,178.0,13.000618,0.811827,11.03,12.3625,13.050,13.6775,14.83
malic acid,178.0,2.336348,1.117146,0.74,1.6025,1.865,3.0825,5.80
ash,178.0,2.366517,0.274344,1.36,2.2100,2.360,2.5575,3.23
alcalinity of ash,178.0,19.494944,3.339564,10.60,17.2000,19.500,21.5000,30.00
magnesium,178.0,99.741573,14.282484,70.00,88.0000,98.000,107.0000,162.00
total phenols,178.0,2.295112,0.625851,0.98,1.7425,2.355,2.8000,3.88
flavanoids,178.0,2.029270,0.998859,0.34,1.2050,2.135,2.8750,5.08
nonflavanoid phenols,178.0,0.361854,0.124453,0.13,0.2700,0.340,0.4375,0.66
proanthocyanins,178.0,1.590899,0.572359,0.41,1.2500,1.555,1.9500,3.58


label,1,2,3
alcohol,13.744746,12.278732,13.153750
malic acid,2.010678,1.932676,3.333750
ash,2.455593,2.244789,2.437083
alcalinity of ash,17.037288,20.238028,21.416667
magnesium,106.338983,94.549296,99.312500
total phenols,2.840169,2.258873,1.678750
flavanoids,2.982373,2.080845,0.781458
nonflavanoid phenols,0.290000,0.363662,0.447500
proanthocyanins,1.899322,1.630282,1.153542
color intensity,5.528305,3.086620,7.396250


- El dataset no tiene valores nulos y las clases están razonablemente equilibradas (59, 71 y 48 vinos).
- Las variables tienen escalas muy distintas (`proline` llega a 1680 y `hue` a 1.71), por lo que conviene escalarlas. 

## Pruebas

Se comparan tres modelos con validación cruzada de 5 particiones, usando solo los datos de entrenamiento.

In [29]:
X = df.drop(columns="label")
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

candidates = {
    "Regresión logística": make_pipeline(
        StandardScaler(), 
        LogisticRegression(max_iter=1000)
    ),
    "KNN": make_pipeline(
        StandardScaler(),
        KNeighborsClassifier()
    ),
    "Random Forest": RandomForestClassifier(random_state=42),
}
for name, candidate in candidates.items():
    scores = cross_val_score(candidate, X_train, y_train, cv=5)
    print(f"{name}: {scores.mean():.3f} | Desviación {scores.std():.3f}")

Regresión logística: 0.993 | Desviación 0.014
KNN: 0.965 | Desviación 0.038
Random Forest: 0.986 | Desviación 0.028


## Modelo final y evaluación

Se elige la regresión logística porque obtiene la mejor media en validación cruzada (0.993) y la menor variación entre particiones. Después se evalúa con los datos de test, que no se han usado hasta ahora.

In [30]:
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
# Entrena el modelo: aprende a reconocer cada tipo de vino a partir de los ejemplos
model.fit(X_train, y_train)
print("Accuracy test:", round(model.score(X_test, y_test), 3))

y_pred = model.predict(X_test)
print("Matriz de confusión (filas: real, columnas: predicción):")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy test: 0.972
Matriz de confusión (filas: real, columnas: predicción):
[[12  0  0]
 [ 0 14  0]
 [ 0  1  9]]
              precision    recall  f1-score   support

           1       1.00      1.00      1.00        12
           2       0.93      1.00      0.97        14
           3       1.00      0.90      0.95        10

    accuracy                           0.97        36
   macro avg       0.98      0.97      0.97        36
weighted avg       0.97      0.97      0.97        36



In [31]:
Path("model").mkdir(exist_ok=True)
joblib.dump(model, "model/wine-model.joblib")

# Comprobación: cargar el archivo y predecir
loaded = joblib.load("model/wine-model.joblib")
print("Predicción:", loaded.predict(X_test.head(3)), "| Real:", y_test.head(3).tolist())

Predicción: [1 2 1] | Real: [1, 3, 1]


## Resultados

El modelo acierta 35 de los 36 vinos de test (accuracy 0.972). Su único error es un vino de tipo 3 clasificado como tipo 2, el mismo que aparece en la comprobación anterior. El resultado es coherente con la validación cruzada (0.993), aunque el conjunto de test es pequeño y cada fallo cambia el accuracy casi 3 puntos.